[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonasneves/colab-slm-playground/blob/main/trending_pytorch_chatbot_colab.ipynb)

# Trending PyTorch Chatbot

Fetches trending `text-generation` models from Hugging Face, lets you pick one, and runs a chat UI inside Colab.

**Requires a GPU runtime.** Go to Runtime → Change runtime type → T4 GPU (or better).

## 1 — Check GPU

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then re-run."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print()
print("Guidelines for this GPU:")
print("  fp16  → models up to ~7B params")
print("  4-bit → models up to ~13B params (lower quality, less memory)")

## 2 — Install dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes huggingface_hub
!wget -q --no-cache https://raw.githubusercontent.com/jonasneves/colab-slm-playground/main/chat_ui.py -O /content/chat_ui.py

## 3 — (Optional) Log in to Hugging Face

Required for gated models (Llama, Gemma, Mistral-instruct, etc.).
Skip this cell if you only plan to use open models.

In [ ]:
# Option A — token stored in Colab secrets (Colab Pro / paid)
# from google.colab import userdata
# from huggingface_hub import login
# login(token=userdata.get("HF_TOKEN"))

# Option B — paste token directly (never commit this)
# from huggingface_hub import login
# login(token="hf_...")

print("Skipped HF login. Uncomment one option above if you need gated models.")

## 4 — Fetch trending models

In [ ]:
from huggingface_hub import HfApi
from chat_ui import build_model_table_html, register_load_callback
from IPython.display import display, HTML
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model = None
tokenizer = None
loaded_model_id = None
PRECISION = "fp16"

print("Fetching trending text-generation models...")

api = HfApi()
raw_models = list(api.list_models(
    pipeline_tag="text-generation",
    sort="trending_score",
    direction=-1,
    limit=40,
    full=True,
))

model_info = []
for m in raw_models:
    libs = m.tags or []
    if "onnx" in libs and "pytorch" not in libs and "transformers" not in libs:
        continue
    gated = getattr(m, "gated", False) or "gated" in libs
    model_info.append({
        "id": m.id,
        "downloads": m.downloads or 0,
        "likes": m.likes or 0,
        "gated": gated,
    })

def load_model(model_id, variant="fp16"):
    global model, tokenizer, loaded_model_id, PRECISION
    PRECISION = variant
    print(f"Loading: {model_id} ({variant})")
    print("Downloading weights — this may take several minutes...")

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

    load_kwargs = dict(device_map="auto", trust_remote_code=True)
    if variant == "4bit":
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
    else:
        load_kwargs["torch_dtype"] = torch.float16

    model = AutoModelForCausalLM.from_pretrained(model_id, **load_kwargs)
    model.eval()
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    loaded_model_id = model_id
    allocated = torch.cuda.memory_allocated() / 1e9
    print(f"Ready. VRAM in use: {allocated:.1f} GB")

register_load_callback(load_model)
display(HTML(build_model_table_html(model_info, precision_toggle=True)))
print(f"Showing {len(model_info)} models.  🔒 = requires HF login.")

## 5 — Launch Chat UI

Select precision (fp16 or 4-bit) and click **Load** in the table above. Once loaded, run this cell.

In [ ]:
from chat_ui import build_chat_html, register_callback
from IPython.display import display, HTML
import torch

assert model is not None, "Click Load in the table above first."

@torch.inference_mode()
def generate(messages: list) -> str:
    def _build_prompt(messages):
        if tokenizer.chat_template:
            return tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        parts = [
            f"{'User' if m['role'] == 'user' else 'Assistant'}: {m['content']}"
            for m in messages
        ]
        return "\n".join(parts) + "\nAssistant:"

    prompt     = _build_prompt(messages)
    inputs     = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    inputs     = {k: v.to(model.device) for k, v in inputs.items()}
    in_len     = inputs["input_ids"].shape[1]

    output_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(output_ids[0][in_len:], skip_special_tokens=True).strip()

register_callback(generate)
short_name      = loaded_model_id.split("/")[-1]
precision_label = "4-bit" if PRECISION == "4bit" else "fp16"
display(HTML(build_chat_html(short_name, f"{precision_label} &middot; GPU")))
print("Chat ready. First response may be slower due to CUDA warm-up.")

---
### Notes

**Why does fp16 fit ~7B parameters on a 16 GB GPU?**
Each parameter stored in fp16 takes 2 bytes. A 7B-parameter model therefore needs ~14 GB just for weights, leaving ~2 GB for activations and the KV cache — tight but workable on a T4. fp32 would require ~28 GB, which exceeds the T4's VRAM entirely. `torch_dtype=torch.float16` with `device_map="auto"` handles this automatically.

**What is 4-bit quantization (NF4)?**
4-bit quantization stores each weight in 4 bits instead of 16, cutting memory roughly 4×. This notebook uses NF4 (NormalFloat4) with double quantization — the scheme introduced by [QLoRA](https://arxiv.org/abs/2305.14314). NF4 is designed specifically for normally-distributed neural network weights, so it preserves more information per bit than generic int4. The result: a 13B model fits in ~7 GB VRAM, at a modest quality cost versus fp16.

**What are gated models?**
Some model authors (Meta for Llama, Google for Gemma) require you to accept a license before downloading. Hugging Face enforces this via gated repos — your account token must be associated with an accepted license. Log in via cell 3, then the download proceeds normally. The 🔒 marker in the table flags these.

**`device_map="auto"`**
This tells `accelerate` to distribute the model across available devices automatically. On a single T4 it simply puts everything on the GPU. On multi-GPU setups or when the model is too large for VRAM alone, it splits layers across GPU and CPU RAM. It's the reason you don't need to manually call `.to("cuda")`.